# DSWx EU Threshold Recalibration Report

This notebook loads the grid-search results from `scripts/recalibrate_dswe_thresholds_results.json`,
plots the F1 surface across the three threshold axes (WIGT / AWGT / PSWT2_MNDWI),
displays the LOO-CV per-fold table (10 folds; B1 fix), reports the held-out Balaton F1
(the official `dswx:eu` matrix-cell value per BOOTSTRAP §5.4), and reproduces the
frozen threshold constants from `src/subsideo/products/dswx_thresholds.py`.

**Phase 6 DSWX-04 + DSWX-05 + DSWX-06 validation:**
- DSWX-04: 8400-gridpoint joint grid search over WIGT × AWGT × PSWT2_MNDWI
- DSWX-05: typed threshold constants module with provenance
- DSWX-06: LOO-CV gap < 0.02 (overfit detection)

PITFALLS P5.1 mitigation: edge-of-grid sentinel + LOO-CV gap acceptance gate enforced
by `scripts/recalibrate_dswe_thresholds.py` before this notebook sees data.

In [ ]:
# Cell 2: Imports
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

print('Imports OK')

In [ ]:
# Cell 3: Load results JSON
results_path = Path('scripts/recalibrate_dswe_thresholds_results.json')
assert results_path.exists(), f'Results file not found: {results_path}'

r = json.loads(results_path.read_text())

print(f"cell_status          : {r['cell_status']}")
print(f"grid_search_run_date : {r.get('grid_search_run_date', 'N/A')}")
print(f"edge_check           : {r['edge_check']['status']}")
print(f"joint_best WIGT      : {r['joint_best_gridpoint']['WIGT']:.4f}  (PROTEUS default: 0.124)")
print(f"joint_best AWGT      : {r['joint_best_gridpoint']['AWGT']:.4f}  (PROTEUS default: 0.0)")
print(f"joint_best PSWT2_MNDWI: {r['joint_best_gridpoint']['PSWT2_MNDWI']:.4f}  (PROTEUS default: -0.5)")
print(f"fit_set_mean_f1      : {r['fit_set_mean_f1']:.4f}")
print(f"loocv_mean_f1        : {r['loocv_mean_f1']:.4f}")
print(f"loocv_gap            : {r['loocv_gap']:.4f}  (threshold: < 0.02 per DSWX-06)")
print(f"held_out_balaton_f1  : {r['held_out_balaton_f1']:.4f}  (OFFICIAL EU matrix-cell value)")
print(f"loocv_per_fold count : {len(r.get('loocv_per_fold', []))}  (expected 10; B1 fix)")

In [ ]:
# Cell 4: Per-pair gridscores ingestion
# Load all 10 fit-set (AOI, season) gridscores.parquet and concatenate (B1 fix: 10 pairs)
FIT_SET_AOIS = ['alcantara', 'tagus', 'vanern', 'garda', 'donana']
CACHE = Path('./eval-dswx-fitset')

dfs = []
for aoi_id in FIT_SET_AOIS:
    for season in ('wet', 'dry'):
        scene_dir = CACHE / aoi_id / season
        if not scene_dir.exists():
            print(f'WARNING: {aoi_id}/{season} cache dir missing')
            continue
        safe_dirs = sorted(scene_dir.glob('S2*.SAFE'))
        if not safe_dirs:
            print(f'WARNING: no SAFE found under {scene_dir}')
            continue
        scene_id = safe_dirs[0].name.replace('.SAFE', '')
        parquet_path = scene_dir / scene_id / 'gridscores.parquet'
        if not parquet_path.exists():
            print(f'WARNING: gridscores.parquet missing for {aoi_id}/{season}')
            continue
        df = pq.read_table(parquet_path).to_pandas()
        df['aoi_id'] = aoi_id
        df['season'] = season
        dfs.append(df)

if dfs:
    all_gs = pd.concat(dfs, ignore_index=True)
    print(f'Loaded {len(dfs)} (AOI, season) gridscores files')
    print(f'Total rows: {len(all_gs):,} (expected {len(dfs) * 8400:,})')
else:
    all_gs = pd.DataFrame()
    print('WARNING: No gridscores loaded (eval-dswx-fitset/ cache empty)')

In [ ]:
# Cell 5: F1 surface plots (3 marginal subplots)
if all_gs.empty:
    print('Skipping F1 surface plots -- no gridscores loaded')
else:
    mean_f1 = all_gs.groupby(['WIGT', 'AWGT', 'PSWT2_MNDWI'])['f1'].mean().reset_index()

    joint_wigt = r['joint_best_gridpoint']['WIGT']
    joint_awgt = r['joint_best_gridpoint']['AWGT']
    joint_pswt2 = r['joint_best_gridpoint']['PSWT2_MNDWI']

    # Marginal mean F1 per axis (averaged over all other axes)
    wigt_marginal = mean_f1.groupby('WIGT')['f1'].mean()
    awgt_marginal = mean_f1.groupby('AWGT')['f1'].mean()
    pswt2_marginal = mean_f1.groupby('PSWT2_MNDWI')['f1'].mean()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    fig.suptitle('DSWx EU Recalibration F1 Surface (marginal averages)', fontsize=13)

    # WIGT axis
    axes[0].plot(wigt_marginal.index, wigt_marginal.values, 'b-o', ms=4)
    axes[0].axvline(joint_wigt, color='r', linestyle='--', label=f'best={joint_wigt:.3f}')
    axes[0].set_xlabel('WIGT'); axes[0].set_ylabel('Mean F1'); axes[0].set_title('WIGT axis')
    axes[0].legend()

    # AWGT axis
    axes[1].plot(awgt_marginal.index, awgt_marginal.values, 'g-o', ms=4)
    axes[1].axvline(joint_awgt, color='r', linestyle='--', label=f'best={joint_awgt:.3f}')
    axes[1].set_xlabel('AWGT'); axes[1].set_ylabel('Mean F1'); axes[1].set_title('AWGT axis')
    axes[1].legend()

    # PSWT2_MNDWI axis
    axes[2].plot(pswt2_marginal.index, pswt2_marginal.values, 'm-o', ms=4)
    axes[2].axvline(joint_pswt2, color='r', linestyle='--', label=f'best={joint_pswt2:.3f}')
    axes[2].set_xlabel('PSWT2_MNDWI'); axes[2].set_ylabel('Mean F1'); axes[2].set_title('PSWT2_MNDWI axis')
    axes[2].legend()

    plt.savefig('notebooks/dswx_recalibration_f1_surface.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Joint best: WIGT={joint_wigt:.4f}, AWGT={joint_awgt:.4f}, PSWT2_MNDWI={joint_pswt2:.4f}')
    print(f'Fit-set mean F1 at joint best: {r["fit_set_mean_f1"]:.4f}')

In [ ]:
# Cell 6: LOO-CV per-fold table (10 rows; B1 fix)
loocv_folds = r.get('loocv_per_fold', [])
print(f'LOO-CV folds: {len(loocv_folds)} (expected 10; B1 fix = leave-one-PAIR-out)')

if loocv_folds:
    loocv_df = pd.DataFrame(loocv_folds)
    loocv_df['test_f1'] = loocv_df['test_f1'].round(4)
    print('\nLOO-CV per-fold table:')
    print(loocv_df[['fold_idx', 'left_out_aoi', 'left_out_season',
                     'refit_best_wigt', 'refit_best_awgt',
                     'refit_best_pswt2_mndwi', 'test_f1']].to_string(index=False))
    print(f'\nLOO-CV mean F1 : {r["loocv_mean_f1"]:.4f}')
    print(f'LOO-CV gap     : {r["loocv_gap"]:.4f}  (acceptance gate: < 0.02 per DSWX-06)')
    gap_ok = r['loocv_gap'] < 0.02
    print(f'Gap gate       : {"PASS" if gap_ok else "FAIL"}')
else:
    print('No LOO-CV folds in results (BLOCKER state?)')

In [ ]:
# Cell 7: Held-out Balaton F1 (OFFICIAL EU matrix-cell value; BOOTSTRAP §5.4 + CONTEXT D-13)
balaton_f1 = r['held_out_balaton_f1']
print(f'Held-out Balaton F1: {balaton_f1:.4f}')
print(f'DSWx F1 gate (BINDING): > 0.90 per DSWX-06 / criteria.py line 188')

if balaton_f1 >= 0.90:
    print(f'Verdict: PASS ({balaton_f1:.4f} >= 0.90)')
    print('Plan 06-07 will report cell_status=PASS for dswx:eu')
elif balaton_f1 >= 0.85:
    print(f'Verdict: FAIL ({balaton_f1:.4f} >= 0.85 but < 0.90)')
    print('Named upgrade path: ML-replacement (DSWX-V2-01)')
else:
    print(f'Verdict: FAIL ({balaton_f1:.4f} < 0.85)')
    print('Named upgrade path: fit-set quality review')

In [ ]:
# Cell 8: Threshold module reproduce
# Confirms that on-disk THRESHOLDS_EU matches the recalibration result
from subsideo.products.dswx_thresholds import THRESHOLDS_EU  # noqa: E402

print(f'THRESHOLDS_EU.WIGT              = {THRESHOLDS_EU.WIGT}')
print(f'THRESHOLDS_EU.AWGT              = {THRESHOLDS_EU.AWGT}')
print(f'THRESHOLDS_EU.PSWT2_MNDWI       = {THRESHOLDS_EU.PSWT2_MNDWI}')
print(f'THRESHOLDS_EU.grid_search_run_date = {THRESHOLDS_EU.grid_search_run_date}')
print(f'THRESHOLDS_EU.fit_set_hash      = {THRESHOLDS_EU.fit_set_hash[:16]}...')
print(f'THRESHOLDS_EU.fit_set_mean_f1   = {THRESHOLDS_EU.fit_set_mean_f1}')
print(f'THRESHOLDS_EU.held_out_balaton_f1 = {THRESHOLDS_EU.held_out_balaton_f1}')
print(f'THRESHOLDS_EU.loocv_mean_f1     = {THRESHOLDS_EU.loocv_mean_f1}')
print(f'THRESHOLDS_EU.loocv_gap         = {THRESHOLDS_EU.loocv_gap}')

# Verify module matches results JSON
if r.get('cell_status') == 'PASS':
    assert abs(THRESHOLDS_EU.WIGT - r['joint_best_gridpoint']['WIGT']) < 1e-4, \
        'WIGT mismatch between module and results JSON'
    print('\nModule matches results JSON: OK')

## Summary

This notebook reports the Phase 6 DSWx EU recalibration outcome.

**Recalibration pipeline:** `scripts/recalibrate_dswe_thresholds.py` ran an
8400-gridpoint joint grid search over WIGT × AWGT × PSWT2_MNDWI on 10 fit-set
(AOI, season) pairs (5 EU biomes × 2 wet/dry seasons; Balaton held out).

**LOO-CV gate (DSWX-06):** 10-fold leave-one-pair-out cross-validation verifies
that the calibration generalises. Gap < 0.02 required.

**Official EU matrix cell:** Held-out Balaton F1 is the OFFICIAL `dswx:eu` value
per BOOTSTRAP §5.4 + CONTEXT D-13. Fit-set mean F1 and LOO-CV mean F1 are
diagnostic-only.

**Threshold module:** `src/subsideo/products/dswx_thresholds.py` `THRESHOLDS_EU`
was updated by Stage 10 of the recalibration script with real grid-search values
and full provenance metadata. Plan 06-07 EU re-run will consume the new thresholds.